# Module 02 - Training and experiment tracking

Train the LightGBM forecaster with shared features and an honest time-series
split, and log to MLflow. See [README.md](README.md).

In [ ]:
# Tutorial setup: find the repo root and make the repo code importable.
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    return start


REPO = find_repo_root(Path.cwd())
for sub in ["src", "data", "webapp/backend"]:
    sys.path.insert(0, str(REPO / sub))
print("Repo root:", REPO)

## Look at the shared feature module

Both training and serving import these columns, which prevents train/serve
skew.

In [ ]:
from common.features import feature_columns

print("Model features:")
for c in feature_columns():
    print(" -", c)

## Train the model

We call the same `train.py` the pipeline uses, with a small `n_estimators` so the
notebook runs quickly.

In [ ]:
import importlib
import mlflow

model_out = REPO / "model_artifacts"
sys.argv = [
    "train.py",
    "--data", str(REPO / "data" / "aeso_hourly.csv"),
    "--model_output", str(model_out),
    "--n_estimators", "200",
]
with mlflow.start_run(run_name="tutorial-02"):
    train = importlib.import_module("training.train")
    train.main()

## Inspect the metrics and model card

In [ ]:
import json

metrics = json.loads((model_out / "metrics.json").read_text())
card = json.loads((model_out / "model_card.json").read_text())
print("Metrics:", json.dumps(metrics, indent=2))
print("Model card name:", card.get("name"))

RMSE is higher than MAE because of scarcity price spikes, the tail risk a
trader cares about. Open the run in AML Studio under Jobs to see it tracked.

## Next

Continue to [Module 03](../03-mlops-pipeline-gate/README.md).